# 手表—皮肤：物理过程与信号示意（4:3）

独立 notebook，不需要先运行其他文件或提供 CSV。选择 `PPG_sensor_env`，从上至下运行即可生成 SVG。
每行左侧是**同一手表、同一个传感点**在三个时刻的剖面，右侧编号标出对应信号时刻。

本图采用 Udeco 的理想化示意，不是样本曲线或标定模型：接触/压迫下降、释放回升，方向沿用当前项目事件约定。
“接触—释放”中的释放表示抬离；“压迫—释放”中的释放表示卸压后恢复轻触。
滑移保持接触并发生切向位移，右侧仅画一种可能扰动，不代表固定模板。

## 只改这里：中文文字、配色、波形
`TEXT` 修改标题和备注；`PROCESSES` 修改各行说明；`STYLE` 修改样式；`SIGNAL` 控制极性和示意幅值。

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path

# ===== 只改这里：中文文字、配色及波形示意 =====
OUTPUT_DIR = PROJECT_ROOT / "figures" / "assets"
TEXT = {
    "title": "手表—皮肤界面：物理过程与信号响应",
    "subtitle": "单点传感器 · 三类过程 · Udeco 理想化示意",
    "physical_header": "物理过程：同一传感点的三个时刻",
    "signal_header": "对应信号：编号与左侧时刻一致",
    "watch": "手表", "sensor": "单点传感器", "skin": "皮肤",
    "signal_axis": "ΔUdeco（示意）", "time_axis": "相对时间",
    "footer": "幅值与时长均为示意；信号极性沿用项目约定，可在配置中翻转。",
    "footer_note": "滑移不存在唯一波形；本图不作为分类模板，也不把算法未分类结果等同于真实滑移。",
}
PROCESSES = [
    {"kind": "contact", "title": "接触—释放", "signal_title": "接触建立 / 解除",
     "phases": ["未接触", "接触", "释放"], "actions": ["靠近", "抬离"],
     "phase_notes": ["存在气隙", "气隙闭合", "再次离开"],
     "physical_note": "物理：间隙闭合，再重新打开。",
     "signal_note": "示意：接触时下降，抬离后回升。", "color": "#B57952"},
    {"kind": "pressure", "title": "压迫—释放", "signal_title": "受力增加 / 卸载",
     "phases": ["轻触", "压迫", "卸压"], "actions": ["加压", "卸载"],
     "phase_notes": ["保持接触", "皮肤压陷", "恢复轻触"],
     "physical_note": "物理：保持接触，法向载荷增减。",
     "signal_note": "示意：渐变加载与卸载，形成较缓起伏。", "color": "#347C9D"},
    {"kind": "slide", "title": "滑移", "signal_title": "切向相对运动",
     "phases": ["位置 A", "滑移中", "位置 B"], "actions": ["切向移动", "继续移动"],
     "phase_notes": ["初始接触位置", "沿皮肤移动", "新接触位置"],
     "physical_note": "物理：接触中发生切向位移。",
     "signal_note": "示意：界面变化引起扰动，形态不固定。", "color": "#388E82"},
]
STYLE = {
    "width": 1600,    # 高度 = width * 3/4，保持整图 4:3
    "font_family": "Microsoft YaHei, Noto Sans CJK SC, SimHei, sans-serif",
    "title_size": 34, "row_title_size": 26, "body_size": 19,
    "ink": "#263C47", "muted": "#6E7F89", "border": "#DDE6EB",
    "skin_fill": "#F5E2D5", "skin_line": "#C49F88",
    "watch_fill": "#E8EEF2", "watch_line": "#7D939F", "sensor_fill": "#334E5D",
}
SIGNAL = {
    "polarity": -1,   # -1：接触/压迫下降，释放回升；+1：整体翻转
    "contact_amplitude": 0.88, "pressure_amplitude": 0.76,
    "slide_amplitude": 0.36,
    "samples": 501,
}


## SVG 与理想化信号函数

仅使用 Python 标准库。文字保留为 SVG `<text>`，图形完全内联，导入矢量编辑软件后仍可修改。
固定三行和 4:3 画布，文字过长会提示精简；改变 `STYLE["width"]` 会整体缩放。
波形的陡/缓区别是本图采用的教学示例；真实响应还取决于动作速度、界面条件和器件动态。

In [ ]:
import math
import unicodedata
from html import escape
from xml.etree import ElementTree as ET


def schematic_signal(kind, t, settings):
    """Dimensionless teaching sketches, not a calibrated sensor or classifier model."""
    def smooth(left, right):
        u = min(1., max(0., (t - left) / (right - left)))
        return u * u * (3 - 2 * u)
    polarity = settings["polarity"]
    if kind == "contact":
        return polarity * settings["contact_amplitude"] * (smooth(.265, .30) - smooth(.70, .735))
    if kind == "pressure":
        return polarity * settings["pressure_amplitude"] * (smooth(.22, .46) - smooth(.58, .83))
    if kind == "slide":
        if not .18 < t < .84:
            return 0.
        u = (t - .18) / .66
        return polarity * settings["slide_amplitude"] * math.sin(math.pi * u) ** 2 * (
            .70 * math.sin(6 * math.pi * u) + .30 * math.sin(13 * math.pi * u + .4))
    raise ValueError(f"未知过程：{kind}")


def build_watch_skin_svg(labels, processes, style, signal):
    """Render a single-point watch/skin cross-section beside each illustrative waveform."""
    if len(processes) != 3 or {p["kind"] for p in processes} != {"contact", "pressure", "slide"}:
        raise ValueError("4:3 图固定包含接触、压迫、滑移三类过程，各一行。")
    if signal["polarity"] not in (-1, 1):
        raise ValueError("polarity 只能为 -1 或 +1。")
    if not isinstance(signal["samples"], int) or signal["samples"] < 101:
        raise ValueError("samples 必须是至少 101 的整数。")
    for key in ("contact_amplitude", "pressure_amplitude", "slide_amplitude"):
        if not math.isfinite(signal[key]) or signal[key] <= 0:
            raise ValueError("示意幅值必须为有限正数。")
    for key in ("width", "title_size", "row_title_size", "body_size"):
        if not math.isfinite(style[key]) or style[key] <= 0:
            raise ValueError("画布尺寸和字号必须为有限正数。")
    s = style
    pieces = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{s["width"]}" '
              f'height="{s["width"]*3/4:g}" viewBox="0 0 1600 1200" role="img" '
              'aria-labelledby="watch-title watch-description">',
              f'<title id="watch-title">{escape(labels["title"])}</title>',
              '<desc id="watch-description">三个物理过程，各含三幅单点传感器剖面和一条理想化信号。</desc>',
              '<defs><marker id="motion-arrow" viewBox="0 0 10 10" refX="9" refY="5" '
              'markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
              '<path d="M 0 0 L 10 5 L 0 10 z" fill="#8C9DA5"/></marker></defs>',
              '<rect width="1600" height="1200" fill="white"/>',
              f'<g font-family="{escape(s["font_family"], quote=True)}">']

    def text(x, y, value, size=19, color=None, weight=400, anchor="start", max_width=None):
        estimated = sum(1 if unicodedata.east_asian_width(c) in "WF" else .62 for c in str(value)) * size
        if max_width is not None and estimated > max_width:
            raise ValueError(f"固定版式的文案过长，请精简或减小字号：{value}")
        pieces.append(f'<text x="{x:.2f}" y="{y:.2f}" font-size="{size}" '
                      f'fill="{color or s["ink"]}" font-weight="{weight}" text-anchor="{anchor}">'
                      f'{escape(str(value))}</text>')

    def rect(x, y, w, h, fill, stroke=None, radius=10, extra=""):
        pieces.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{w:.2f}" height="{h:.2f}" '
                      f'rx="{radius}" fill="{fill}" stroke="{stroke or s["border"]}" stroke-width="1.2" {extra}/>')

    def line(points, color=None, width=1.5, dashed=False, arrow=False):
        coords = ' '.join(f'{x:.2f},{y:.2f}' for x, y in points)
        pieces.append(f'<polyline points="{coords}" fill="none" stroke="{color or s["border"]}" '
                      f'stroke-width="{width}" stroke-linecap="round" stroke-linejoin="round" '
                      + ('stroke-dasharray="5 5" ' if dashed else '')
                      + ('marker-end="url(#motion-arrow)" ' if arrow else '') + '/>')

    def circle(x, y, radius, fill, stroke=None, extra=""):
        pieces.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="{radius}" '
                      f'fill="{fill}" stroke="{stroke or fill}" stroke-width="1.5" {extra}/>')

    def badge(x, y, number, color):
        circle(x, y, 11, 'white', color)
        text(x, y + 5, number, 15, color, 600, "middle")

    def device(cx, skin_y, gap, depth, shift, color, sliding=False):
        """One sensor per pose. Skin frame stays fixed as the watch translates."""
        sensor_x = cx + shift
        surface = [(cx - 84 + i * 168 / 48,
                    skin_y + depth * math.exp(-((cx - 84 + i * 168 / 48 - sensor_x) / 34) ** 2))
                   for i in range(49)]
        polygon = surface + [(cx + 84, skin_y + 34), (cx - 84, skin_y + 34)]
        coords = ' '.join(f'{x:.2f},{y:.2f}' for x, y in polygon)
        pieces.append(f'<polygon points="{coords}" fill="{s["skin_fill"]}"/>')
        line(surface, s["skin_line"], 1.8)
        for offset in (-56, -28, 0, 28, 56):
            circle(cx + offset, skin_y + 23, 1.6, '#D8BCA9')
        # Each pose uses one solid dot on the underside of the case.
        sensor_bottom = skin_y + depth - gap
        case_bottom = sensor_bottom - 14
        rect(sensor_x - 67, case_bottom - 25, 134, 14, s["watch_fill"], s["watch_line"], 4)
        rect(sensor_x - 46, case_bottom - 36, 92, 36, s["watch_fill"], s["watch_line"], 9)
        rect(sensor_x - 30, case_bottom - 29, 60, 15, 'white', 'none', 4)
        circle(sensor_x, sensor_bottom - 7, 7, s["sensor_fill"], extra='data-sensor="single"')
        if gap > 0:
            line([(sensor_x + 57, sensor_bottom + 2), (sensor_x + 57, skin_y - 2)], color, 1.2, dashed=True)
        if sliding:
            line([(cx, skin_y + 5), (cx, skin_y + 34)], s["skin_line"], 1.2, dashed=True)

    text(48, 60, labels["title"], s["title_size"], weight=700, max_width=1504)
    text(48, 99, labels["subtitle"], 21, s["muted"], max_width=890)
    # Compact legend: all three phases represent the same device, not three sensors.
    for x, name, fill in [(1030, labels["watch"], s["watch_fill"]),
                           (1170, labels["sensor"], s["sensor_fill"]),
                           (1390, labels["skin"], s["skin_fill"])]:
        if name == labels["sensor"]:
            circle(x + 9, 93, 6, fill)
        else:
            rect(x, 85, 22, 15, fill, s["border"], 3)
        text(x + 31, 100, name, 18, s["muted"], max_width=185)
    text(68, 151, labels["physical_header"], 21, weight=600, max_width=700)
    text(850, 151, labels["signal_header"], 21, weight=600, max_width=650)

    times = [i / (signal["samples"] - 1) for i in range(signal["samples"])]
    phase_times = (.12, .50, .88)
    amplitudes = max(signal["contact_amplitude"], signal["pressure_amplitude"], signal["slide_amplitude"])
    scale = 87 / amplitudes  # One common amplitude scale; supports polarity reversal without clipping.
    for row, process in enumerate(processes):
        if len(process["phases"]) != 3 or len(process["phase_notes"]) != 3 or len(process["actions"]) != 2:
            raise ValueError("每个过程需要三个阶段、三个阶段备注和两个动作标签。")
        top = 178 + row * 312
        color, kind = process["color"], process["kind"]
        pieces.append(f'<g data-process="{kind}">')
        rect(48, top, 1504, 296, '#FBFCFD', radius=18)
        rect(68, top + 19, 5, 28, color, 'none', 2)
        text(88, top + 40, f'{"abc"[row]}  {process["title"]}', s["row_title_size"], color, 600, max_width=655)
        text(850, top + 39, process["signal_title"], 22, color, 600, max_width=650)
        line([(800, top + 62), (800, top + 255)], s["border"], 1)
        centers = (182, 410, 638)
        if kind == 'contact':
            poses = [(28, 0, 0), (0, 0, 0), (28, 0, 0)]
        elif kind == 'pressure':
            poses = [(0, 3, 0), (0, 17, 0), (0, 3, 0)]
        else:
            poses = [(0, 3, -18), (0, 3, 0), (0, 3, 18)]
        for j, (cx, pose) in enumerate(zip(centers, poses)):
            badge(cx - 58, top + 69, str(j + 1), color)
            text(cx - 36, top + 75, process["phases"][j], 19, weight=600, max_width=125)
            device(cx, top + 173, *pose, color, sliding=kind == 'slide')
            text(cx, top + 229, process["phase_notes"][j], 18, s["muted"], anchor='middle', max_width=180)
        for j, action in enumerate(process["actions"]):
            left, right = centers[j] + 90, centers[j + 1] - 90
            line([(left, top + 128), (right, top + 128)], color, 1.6, arrow=True)
            text((left + right) / 2, top + 109, action, 16, s["muted"], anchor='middle', max_width=140)
        if kind == 'pressure':
            # Normal force on the watch, ending above the compressed watch case.
            line([(410, top + 92), (410, top + 132)], color, 2.6, arrow=True)
        if kind == 'slide':
            line([(384, top + 106), (442, top + 106)], color, 2.6, arrow=True)
        # Schematic signals. Numbered time markers link explicitly to the three poses.
        x0, pw = 860, 635
        zero_y = top + (199 if signal['polarity'] == 1 else 112)
        def point(t):
            return x0 + pw * t, zero_y - scale * schematic_signal(kind, t, signal)
        line([(x0, zero_y), (x0 + pw, zero_y)], '#C5D0D6', 1, dashed=True)
        text(x0 - 10, zero_y + 6, '0', 15, s["muted"], anchor='end')
        for j, t in enumerate(phase_times):
            x, y = point(t)
            line([(x, top + 89), (x, top + 222)], '#E0E7EB', 1, dashed=True)
            badge(x, top + 72, str(j + 1), color)
        line([point(t) for t in times], color, 3)
        for t in phase_times:
            x, y = point(t)
            circle(x, y, 4, color)
        line([(x0, top + 235), (x0 + pw, top + 235)], '#C4D0D6', 1.1, arrow=True)
        text(x0 + pw, top + 256, labels["time_axis"], 16, s["muted"], anchor='end')
        text(x0, top + 256, labels["signal_axis"], 16, s["muted"], max_width=380)
        text(88, top + 267, process["physical_note"], s["body_size"], s["muted"], max_width=655)
        # Keep the waveform interpretation beside its row heading, not beneath the axis.
        text(850, top + 285 - 1, process["signal_note"], 17, s["muted"], max_width=650)
        pieces.append('</g>')
    text(48, 1140, labels["footer"], 19, s["muted"], max_width=1504)
    text(48, 1173, labels["footer_note"], 18, s["muted"], max_width=1504)
    pieces.extend(['</g>', '</svg>'])
    svg = '\n'.join(pieces)
    ET.fromstring(svg)
    return svg


## 生成 SVG 并预览

In [ ]:
from IPython.display import SVG, display

svg_text = build_watch_skin_svg(TEXT, PROCESSES, STYLE, SIGNAL)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
svg_path = OUTPUT_DIR / "watch_skin_process_signals.svg"
svg_path.write_text(svg_text, encoding="utf-8")
display(SVG(data=svg_text))
print(f"已生成：{svg_path}")

## 修改与使用

- 直接修改顶部中文字符串，再依次运行配置、函数和生成三个单元格。
- `SIGNAL["polarity"] = -1` 表示接触/压迫下降；改成 `+1` 可以翻转曲线，但需同步修改各行的方向说明。
- 若要改变具体波形形态，编辑 `schematic_signal()`：接触采用较短过渡，压迫采用平缓过渡，滑移采用局部多相扰动。
- `OUTPUT_DIR` 控制 SVG 保存目录。直接编辑 SVG 的改动会被下一次生成覆盖。
- 这里的三个时刻只是为理解物理过程选取的快照，不对应精确力值、距离或持续时间。
- 本图不运行分类器；也不把现有代码中 `cancelled → slide` 的展示别名当作真实滑移标签。

### 参考与信号方向来源

接近/接触压力可改变传感器—皮肤界面条件，相关界面传感研究见
[Wang et al., 2021](https://pubmed.ncbi.nlm.nih.gov/34030098/)。该文提供物理背景，不用于证明本图的示意波形或 Udeco 极性。
本图极性来自当前项目的事件映射：`attach/force` 对应负方向，`detach/release` 对应正方向；曲线为独立生成的教学示意。